In [3]:
# Data Integration Notebook
# This notebook integrates the three cleaned datasets

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [4]:
# Create processed directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

print("Loading datasets...")
# Load the one-hot encoded crime data
crime_data = pd.read_csv('../data/processed/crime_data_311.csv')
print(f"Loaded crime data: {crime_data.shape}")


Loading datasets...
Loaded crime data: (3043, 79)


In [5]:
# Load cleaned ACS data
acs_data = pd.read_csv('../data/processed/demographics_data_cleaned.csv')
print(f"\nLoaded ACS data: {acs_data.shape[0]} rows, {acs_data.shape[1]} columns")
print(f"ACS data years: {sorted(acs_data['year'].unique())}")
print(f"Number of unique ZIP codes in ACS data: {acs_data['zip_code'].nunique()}")

# Load cleaned business data
business_data = pd.read_csv('../data/processed/business_data_cleaned.csv')
print(f"\nLoaded business data: {business_data.shape[0]} rows, {business_data.shape[1]} columns")
print(f"Business data years: {sorted(business_data['year'].unique())}")
print(f"Number of unique ZIP codes in business data: {business_data['zip_code'].nunique()}")



Loaded ACS data: 2348 rows, 28 columns
ACS data years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Number of unique ZIP codes in ACS data: 214

Loaded business data: 420 rows, 134 columns
Business data years: [np.int64(2022), np.int64(2023), np.int64(2024)]
Number of unique ZIP codes in business data: 141


In [6]:
# Standardize ZIP code columns
print("Standardizing ZIP code columns...")
# Rename ZCTA to zip_code in ACS data
acs_data = acs_data.rename(columns={'zcta': 'zip_code'})

# Rename first column in business data if it's unnamed
if business_data.columns[0] == '':
    business_data = business_data.rename(columns={'': 'zip_code'})
# Rename year column in business data for consistency
business_data = business_data.rename(columns={'yr': 'year'})

# Find common ZIP codes
crime_zips = set(crime_data['zip_code'].unique())
acs_zips = set(acs_data['zip_code'].unique())
business_zips = set(business_data['zip_code'].unique())

common_zips = crime_zips.intersection(acs_zips).intersection(business_zips)
print(f"Number of common ZIP codes: {len(common_zips)}")

Standardizing ZIP code columns...
Number of common ZIP codes: 140


In [7]:
# Find common years
crime_years = set(crime_data['year'].unique())
acs_years = set(acs_data['year'].unique())
business_years = set(business_data['year'].unique())

common_years = crime_years.intersection(acs_years).intersection(business_years)
print(f"Common years: {sorted(common_years)}")

Common years: [np.int64(2022), np.int64(2023)]


In [8]:
# Filter each dataset to common ZIP codes and years
crime_filtered = crime_data[
    (crime_data['zip_code'].isin(common_zips)) & 
    (crime_data['year'].isin(common_years))
]

acs_filtered = acs_data[
    (acs_data['zip_code'].isin(common_zips)) & 
    (acs_data['year'].isin(common_years))
]

business_filtered = business_data[
    (business_data['zip_code'].isin(common_zips)) & 
    (business_data['year'].isin(common_years))
]

In [9]:
print(f"Filtered crime data shape: {crime_filtered.shape}")
print(f"Filtered ACS data shape: {acs_filtered.shape}")
print(f"Filtered business data shape: {business_filtered.shape}")

Filtered crime data shape: (280, 79)
Filtered ACS data shape: (280, 28)
Filtered business data shape: (279, 134)


In [10]:
# Merge all three datasets
print("Merging all datasets...")
merged_data = pd.merge(
    crime_filtered,
    acs_filtered,
    on=['zip_code', 'year'],
    how='inner'
)

merged_data = pd.merge(
    merged_data,
    business_filtered,
    on=['zip_code', 'year'],
    how='inner'
)

print(f"Final merged dataset shape: {merged_data.shape}")

Merging all datasets...
Final merged dataset shape: (279, 237)


In [11]:
# Save the integrated dataset
merged_data.to_csv('../data/processed/integrated_all_columns.csv', index=False)
print("Saved merged dataset to '../data/processed/integrated_all_columns.csv'")


Saved merged dataset to '../data/processed/integrated_all_columns.csv'


In [12]:
# Print column counts by source
crime_cols = len(crime_filtered.columns) - 2  # Subtract zip_code and year
acs_cols = len(acs_filtered.columns) - 2
business_cols = len(business_filtered.columns) - 2
total_cols = len(merged_data.columns)

print(f"\nColumn counts in final dataset:")
print(f"Crime data columns: {crime_cols}")
print(f"ACS data columns: {acs_cols}")
print(f"Business data columns: {business_cols}")
print(f"Total columns: {total_cols}")

# Print sample
print("\nSample of merged data (first 3 rows, selected columns):")
selected_cols = ['zip_code', 'year']
selected_cols.extend([col for col in merged_data.columns if col not in ['zip_code', 'year']][:5])  # Add first 5 non-key columns
print(merged_data[selected_cols].head(3))


Column counts in final dataset:
Crime data columns: 77
ACS data columns: 26
Business data columns: 132
Total columns: 237

Sample of merged data (first 3 rows, selected columns):
   zip_code  year  After Hours - Licensed Est  Animal Waste  Banging/Pounding  \
0     10001  2022                         9.0           NaN             639.0   
1     10002  2022                         4.0           NaN             990.0   
2     10003  2022                         1.0           NaN             410.0   

   Blocked Bike Lane  Blocked Crosswalk  
0              255.0               67.0  
1              884.0              224.0  
2              513.0               81.0  
